# Active Impedance Modulation & Zero-Downtime AI Control for Long-Haul Transmission Harmonic & SSR Mitigation

**Lead Author & Principal Investigator:** Abhishek Singh ([ORCID: 0009-0004-6913-096X](https://orcid.org/0009-0004-6913-096X))  
**AI Computational & Multi-Physics Support:** Team Gemini (Google DeepMind)  
**Repository:** [Abhishek1033ubuntu/long-haul-ssr-harmonic-mitigation](https://github.com/Abhishek1033ubuntu/long-haul-ssr-harmonic-mitigation)

---

### Interactive Demonstration Overview
This notebook compiles and executes the bit-exact `float32` C++ HLS FPGA kernel (`rtl_hls/redundant_grid_defense_top.cpp`) natively inside Google Colab's Ubuntu container using `g++`, validating:
1. **Sub-Synchronous Control Interaction (SSCI) Quenching:** Rapid damping of a 25 Hz negative-damping contingency (+14.0 s⁻¹) on a 400 kV, 500 km corridor.
2. **4-Node Cross-Coupling Matrix (`W_decouple`):** Counter-phase reactive trims across distributed substations to eliminate traveling-wave reflections.
3. **Zero-Downtime Redundancy & Live Maintenance:** Concurrent 100% LAN-A fiber ring cut (IEC 62439-3 PRP) and live Primary Board A power-off handoff (5.0 ns switchover) with zero transient bump.
4. **Hardware Crowbar Surge Interlock:** Immediate bypass trip during a 32 kA three-phase bolted fault.

In [ ]:
# Step 1: Clone the GitHub repository into the Colab workspace
!git clone https://github.com/Abhishek1033ubuntu/long-haul-ssr-harmonic-mitigation.git
%cd long-haul-ssr-harmonic-mitigation
!ls -la

In [ ]:
# Step 2: Compile the C++ FPGA HLS Kernel with GCC (-O3 -std=c++17) and execute the 15,000-cycle verification
!python3 testbench/verify_corridor_sim.py

In [ ]:
# Step 3: Run Full Telemetry Logging & Render High-Resolution Diagnostic Waveforms
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Build a CSV-exporting wrapper around the repository's HLS headers
wrapper_code = '''
#include <fstream>
#include <random>
#include "rtl_hls/grid_defense_types.hpp"
#include "rtl_hls/ekf_damping_core.hpp"

// Forward declaration from redundant_grid_defense_top.cpp (compiled without main)
void redundant_grid_defense_top(
    const NodeTelemetry lan_a_telem[NUM_NODES],
    const NodeTelemetry lan_b_telem[NUM_NODES],
    uint64_t master_ptp_ns,
    bool board_a_heartbeat_ok,
    bool board_b_heartbeat_ok,
    bool maint_switch_pull_a,
    bool maint_switch_pull_b,
    float i_nom_init,
    float thresh_on,
    float thresh_off,
    float crowbar_limit,
    NodeHardwareOutputs final_optical_gates[NUM_NODES],
    RedundantSystemStatus &sys_status
);

int main() {
    const double dt = 1e-4;
    const int steps = 15000;
    const double omega0 = 2.0 * M_PI * 50.0;
    const double omega_sso = 2.0 * M_PI * 25.0;

    std::mt19937 gen(42);
    std::normal_distribution<double> noise_v(0.0, 0.05);
    std::normal_distribution<double> noise_i(0.0, 0.003);

    double A_sso_open = 0.012, A_sso_n1 = 0.012, phi_sso = 0.0;
    float prev_dL[NUM_NODES] = {0.0f}, prev_dC[NUM_NODES] = {0.0f};

    std::ofstream csv("demo_results.csv");
    csv << "t,ia_open,ia_closed,A_true,A_est,dC_n1,dC_n2,lan_a_ok,active_primary,maint_safe,crowbar_n2\n";

    for (int k = 0; k < steps; k++) {
        double tk = k * dt;
        uint64_t master_ptp_ns = static_cast<uint64_t>(k) * 100000ULL;

        double sigma_n1 = (tk >= 0.20 && tk < 1.15) ? 14.0 : -8.0;
        A_sso_open = std::min(18.0, A_sso_open * std::exp(sigma_n1 * dt));
        double mod_effort = std::sqrt(std::pow(prev_dL[0] / 0.30, 2) + std::pow(prev_dC[0] / 0.30, 2));
        A_sso_n1 = std::max(0.008, A_sso_n1 * std::exp((sigma_n1 - 36.0 * mod_effort) * dt));
        phi_sso = std::fmod(phi_sso + omega_sso * dt, 2.0 * M_PI);

        bool lan_a_up     = !(tk >= 0.45 && tk < 0.85);
        bool maint_pull_a =  (tk >= 0.60 && tk < 0.95);
        bool board_a_hb   = !(tk >= 0.62 && tk < 0.93);
        double fault_n2   =  (tk >= 1.25 && tk < 1.30) ? 31.5 : 0.0;

        NodeTelemetry lan_a[NUM_NODES], lan_b[NUM_NODES];
        double atten[NUM_NODES] = {1.0, 0.35, 0.12, 0.04};
        double id_n1 = 1.5, iq_n1 = 0.2;

        for (int i = 0; i < NUM_NODES; i++) {
            double local_sso = A_sso_n1 * atten[i];
            double local_fault = (i == 1) ? fault_n2 : 0.0;
            double id = 1.5 + local_fault + local_sso * std::cos(phi_sso);
            double iq = 0.2 + local_sso * std::sin(phi_sso);
            if (i == 0) { id_n1 = id; iq_n1 = iq; }

            NodeTelemetry pkt;
            pkt.vd_meas = static_cast<float>(326.5 - (local_fault > 0 ? 180.0 : 0.0) - 12.0 * local_sso * std::cos(phi_sso) + noise_v(gen));
            pkt.vq_meas = static_cast<float>(-12.0 * local_sso * std::sin(phi_sso) + noise_v(gen));
            pkt.id_meas = static_cast<float>(id + noise_i(gen));
            pkt.iq_meas = static_cast<float>(iq + noise_i(gen));
            pkt.grid_phase_rad = static_cast<float>(std::fmod(omega0 * tk, 2.0 * M_PI));
            pkt.ptp_timestamp_ns = master_ptp_ns + 120ULL;
            lan_a[i] = pkt; lan_a[i].valid_frame = lan_a_up;
            lan_b[i] = pkt; lan_b[i].valid_frame = true;
        }

        NodeHardwareOutputs gates[NUM_NODES];
        RedundantSystemStatus status;
        redundant_grid_defense_top(
            lan_a, lan_b, master_ptp_ns, board_a_hb, true, maint_pull_a, false,
            1.5f, 0.050f, 0.015f, 25.0f, gates, status
        );

        for (int i = 0; i < NUM_NODES; i++) {
            prev_dL[i] = gates[i].delta_L_coord;
            prev_dC[i] = gates[i].delta_C_coord;
        }

        double ia_open = 1.5 * std::cos(omega0 * tk) - 0.2 * std::sin(omega0 * tk) + A_sso_open * std::cos(omega_sso * tk + phi_sso);
        double ia_closed = id_n1 * std::cos(omega0 * tk) - iq_n1 * std::sin(omega0 * tk);

        csv << tk << "," << ia_open << "," << ia_closed << "," << A_sso_n1 << "," << gates[0].est_A_sso << ","
            << gates[0].delta_C_coord << "," << gates[1].delta_C_coord << ","
            << (status.lan_a_healthy ? 1 : 0) << "," << (status.active_is_primary ? 1 : 0) << ","
            << (status.maintenance_safe_ok ? 1 : 0) << "," << (gates[1].bypass_crowbar_cmd ? 1 : 0) << "\n";
    }
    csv.close();
    return 0;
}
'''

with open("demo_driver.cpp", "w") as f:
    f.write(wrapper_code)

subprocess.run(["g++", "-O3", "-std=c++17", "-D__SYNTHESIS__", "rtl_hls/redundant_grid_defense_top.cpp", "demo_driver.cpp", "-o", "demo_bin"], check=True)
subprocess.run(["./demo_bin"], check=True)

df = pd.read_csv("demo_results.csv")

fig, axs = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axs[0].plot(df['t'], df['ia_open'], color='#d62728', alpha=0.55, label='Uncontrolled Corridor Current (kA)')
axs[0].plot(df['t'], df['ia_closed'], color='#1f77b4', label='FPGA AI-Controlled Corridor Current (kA)')
axs[0].set_ylabel("Current (kA)")
axs[0].set_title("Long-Haul 400 kV Corridor: SSCI Quenching, Multi-Node Trim & Zero-Downtime Redundancy", fontweight="bold")
axs[0].legend(loc="upper left")
axs[0].grid(True, alpha=0.4)

axs[1].plot(df['t'], df['A_true'], color='#2ca02c', linewidth=1.6, label='True 25 Hz SSO Amplitude (kA)')
axs[1].plot(df['t'], df['A_est'], 'k--', linewidth=1.0, label='FPGA float32 EKF Estimate (kA)')
axs[1].axvspan(0.45, 0.85, color='#ffbb78', alpha=0.25, label='LAN-A Fiber Cut')
axs[1].axvspan(0.60, 0.95, color='#aec7e8', alpha=0.35, label='Board A Live Maintenance Pull')
axs[1].set_ylabel("SSO Amp (kA)")
axs[1].set_ylim(0, 0.065)
axs[1].legend(loc="upper right", ncol=2)
axs[1].grid(True, alpha=0.4)

axs[2].step(df['t'], df['lan_a_ok'], color='#ff7f0e', where='post', label='LAN-A Health (1=OK, 0=Cut)')
axs[2].step(df['t'], df['active_primary']*0.85, color='#9467bd', where='post', linewidth=1.5, label='Bus Master (0.85=Board A, 0=Board B)')
axs[2].step(df['t'], df['maint_safe']*0.70, color='#17becf', where='post', linestyle='--', label='SAFE-TO-SERVICE LED')
axs[2].step(df['t'], df['crowbar_n2']*0.50, color='#d62728', where='post', linewidth=1.5, label='Node 2 Crowbar Trip (32 kA Fault)')
axs[2].set_ylabel("Redundancy Flags")
axs[2].set_xlabel("Time (seconds)")
axs[2].set_ylim(-0.1, 1.15)
axs[2].legend(loc="lower left", ncol=2)
axs[2].grid(True, alpha=0.4)
plt.tight_layout()
plt.show()